# 034 — ESI ion-current estimator (Rhodamine 6G)

Upper-bound estimate: **if every analyte molecule leaves the emitter as an ion**,
what current does that equal at a given flow rate?

$$I = z \, F \, c \, Q$$

- $c$ — analyte concentration [mol/L], $Q$ — flow rate [L/s], $F = e N_A$ = 96485 C/mol
- $z$ — charges per ion. R6G is sold as the chloride salt; the cation R6G⁺ (m/z 443.2) is
  **pre-charged**, so $z = 1$ needs no protonation. $z = 2$ ≙ [M+H]²⁺.

**0.1 % formic acid toggle** adds the protons from formic-acid dissociation as a second
charge carrier (weak acid, quadratic equilibrium with aqueous pKa 3.75). Those protons
outnumber 10 µM R6G by ~200×, so the "everything ionized" total jumps from nA to tens of µA.

**Reality check:** a real ESI spray carries only ~10 nA – 1 µA, set by spray physics
(conductivity, surface tension, flow), not by ion supply. Once the ion supply exceeds the
spray current, carriers compete for the excess charge. The last output line gives a crude
share: R6G gets $z c_{R6G} / (z c_{R6G} + [\mathrm{H^+}])$ of an assumed spray current.
No hardware is touched in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display

F = 96485.332          # C/mol, Faraday constant
N_A = 6.02214076e23    # 1/mol

# Formic acid additive (0.1 % v/v)
FA_VOL_FRACTION = 0.001      # 0.1 % v/v
FA_DENSITY = 1.22            # g/mL, neat formic acid
FA_MOLAR_MASS = 46.025       # g/mol
FA_PKA = 3.75                # aqueous; in MeOH/water mixtures pKa is higher (fewer free H+)

C_FA = FA_VOL_FRACTION * FA_DENSITY * 1000 / FA_MOLAR_MASS   # mol/L  (~26.5 mM)


def proton_conc(c_acid, pka=FA_PKA):
    # [H+] of a weak acid: x^2 / (c - x) = Ka  ->  positive root
    ka = 10 ** -pka
    return (-ka + np.sqrt(ka * ka + 4 * ka * c_acid)) / 2


def ion_current(c_mol_per_l, flow_ul_per_min, z=1):
    # I = z F c Q, returns ampere
    q_l_per_s = flow_ul_per_min * 1e-6 / 60
    return z * F * c_mol_per_l * q_l_per_s


def fmt_current(i):
    for scale, unit in ((1e-3, "mA"), (1e-6, "µA"), (1e-9, "nA"), (1e-12, "pA")):
        if abs(i) >= scale:
            return f"{i / scale:8.3f} {unit}"
    return f"{i / 1e-15:8.3f} fA"


# Sanity: 10 µM, 10 µL/min, z=1 -> 160.8 nA
assert abs(ion_current(1e-5, 10) - 160.8e-9) < 0.1e-9
H_FA = proton_conc(C_FA)
print(f"0.1 % FA: c = {C_FA*1e3:.1f} mM, [H+] = {H_FA*1e3:.2f} mM (pH {-np.log10(H_FA):.2f}, "
      f"{H_FA/C_FA*100:.1f} % dissociated)")

0.1 % FA: c = 26.5 mM, [H+] = 2.08 mM (pH 2.68, 7.9 % dissociated)


In [2]:
conc = w.FloatLogSlider(value=1e-5, base=10, min=-9, max=-2, step=0.05,
                        description="c R6G [mol/L]", readout_format=".2e",
                        style={"description_width": "120px"}, layout=w.Layout(width="520px"))
flow = w.FloatSlider(value=10.0, min=0.1, max=50.0, step=0.1,
                     description="Flow [µL/min]", readout_format=".1f",
                     style={"description_width": "120px"}, layout=w.Layout(width="520px"))
charge = w.ToggleButtons(options=[("z = 1  (R6G⁺)", 1), ("z = 2  ([M+H]²⁺)", 2)], value=1,
                         description="Charge", style={"description_width": "120px"})
formic = w.ToggleButton(value=False, description="0.1 % formic acid: OFF",
                        icon="flask", layout=w.Layout(width="250px"))
spray = w.FloatLogSlider(value=100e-9, base=10, min=-9, max=-5, step=0.05,
                         description="Spray I [A]", readout_format=".1e",
                         style={"description_width": "120px"}, layout=w.Layout(width="520px"))


def _label_formic(change):
    formic.description = f"0.1 % formic acid: {'ON' if change['new'] else 'OFF'}"
    formic.button_style = "success" if change["new"] else ""


formic.observe(_label_formic, names="value")


def show(c, q, z, fa, i_spray):
    i_r6g = ion_current(c, q, z)
    h = proton_conc(C_FA) if fa else 0.0
    i_h = ion_current(h, q, 1)
    i_total = i_r6g + i_h
    rate = c * q * 1e-6 / 60 * N_A

    print(f"c = {c:.3e} mol/L   Q = {q:.1f} µL/min   z = {z}   formic acid: {'yes' if fa else 'no'}")
    print(f"R6G ions             {rate:10.3e} 1/s   ->  I = {fmt_current(i_r6g)}")
    if fa:
        print(f"H+ from 0.1 % FA     {h*q*1e-6/60*N_A:10.3e} 1/s   ->  I = {fmt_current(i_h)}"
              f"   ([H+] = {h*1e3:.2f} mM)")
        print(f"Total (all ionized)                       I = {fmt_current(i_total)}")
        print(f"  (FA fully dissociated upper bound: {fmt_current(ion_current(C_FA, q))})")
    share = z * c / (z * c + h)
    print(f"\nCrude R6G share of a {fmt_current(i_spray).strip()} spray: {share*100:5.1f} %"
          f"  ->  {fmt_current(min(i_spray * share, i_r6g)).strip()} R6G current"
          f"{'  (ion-supply limited)' if i_spray * share > i_r6g else ''}")

    qs = np.linspace(0.1, 50, 300)
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    ax.plot(qs, ion_current(c, qs, z) * 1e9, label=f"R6G (z={z})")
    if fa:
        ax.plot(qs, ion_current(h, qs, 1) * 1e9, label="H⁺ from 0.1 % FA")
        ax.plot(qs, (ion_current(c, qs, z) + ion_current(h, qs, 1)) * 1e9, "k--", lw=1, label="total")
    ax.axhline(i_spray * 1e9, color="grey", ls=":", label="assumed spray current")
    ax.plot([q], [i_r6g * 1e9], "o", color="C0")
    ax.set_yscale("log")
    ax.set_xlabel("Flow rate [µL/min]")
    ax.set_ylabel("Current [nA]")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


out = w.interactive_output(show, {"c": conc, "q": flow, "z": charge, "fa": formic, "i_spray": spray})
display(w.VBox([conc, flow, charge, formic, spray]), out)

Output()